# Book Recommender Project

Book Scrapped from [goodreads](https://www.goodreads.com/book/popular_by_date/2019).

Developed by: Bryan Calderon

Extracting the 10 most popular books per year!

#### Scrapping

* Loading libraries

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import urljoin

* Defining the entry

In [16]:
base_url = "https://www.goodreads.com"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9"
}

* Preparing functions to get soup and to extract books by year

In [17]:
def get_soup(url, headers=headers, retries=3, sleep_time=3):
    """
    Requests a page and returns a BeautifulSoup object.
    If the request fails, it retries a few times before returning None.
    """
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=headers, timeout=20)
            response.raise_for_status()
            return BeautifulSoup(response.text, "html.parser")
        
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1} failed for {url}")
            print(e)
            time.sleep(sleep_time)
    
    return None

In [24]:
def extract_books_from_year_page(year, books_per_year=10):
    """
    Extracts only likely book URLs from Goodreads popular books by year page.
    """
    url = f"https://www.goodreads.com/book/popular_by_date/{year}"
    soup = get_soup(url)
    
    if soup is None:
        return []
    
    links = soup.find_all("a", href=True)
    
    books = []
    seen_ids = set()
    
    for link in links:
        href = link["href"]
        text = link.get_text(" ", strip=True)
        
        if "/book/show/" not in href:
            continue
        
        # Remove empty or suspicious link texts
        if not text or len(text) < 3:
            continue
        
        # Skip ISBN-like numeric links
        if text.isdigit():
            continue
        
        # Skip weak generic texts
        bad_texts = {"here", "here.", "more", "less", "preview", "read more",
        "kindle", "amazon", "audible", "worldcat", "libraries", "ACE#1", "ACE #1"}
        if text.lower() in bad_texts:
            continue
        
        # Extract Goodreads book ID to avoid duplicates
        match = re.search(r"/book/show/(\d+)", href)
        
        if not match:
            continue
        
        book_id = match.group(1)
        
        if book_id in seen_ids:
            continue
        
        seen_ids.add(book_id)
        
        book_url = urljoin(base_url, href.split("?")[0])
        
        books.append({
            "source_year": year,
            "rank": len(books) + 1,
            "list_title": text,
            "book_url": book_url,
            "source": "Goodreads Popular by Year"
        })
        
        if len(books) == books_per_year:
            break
    
    return books

* Testing with 2024

In [25]:
books_2024 = extract_books_from_year_page(2023, books_per_year=10)
books_2024

[{'source_year': 2023,
  'rank': 1,
  'list_title': 'Fourth Wing (The Empyrean, #1)',
  'book_url': 'https://www.goodreads.com/book/show/61431922-fourth-wing',
  'source': 'Goodreads Popular by Year'},
 {'source_year': 2023,
  'rank': 2,
  'list_title': 'Iron Flame (The Empyrean, #2)',
  'book_url': 'https://www.goodreads.com/book/show/90202302-iron-flame',
  'source': 'Goodreads Popular by Year'},
 {'source_year': 2023,
  'rank': 3,
  'list_title': "The Housemaid's Secret (The Housemaid, #2)",
  'book_url': 'https://www.goodreads.com/book/show/62848145-the-housemaid-s-secret',
  'source': 'Goodreads Popular by Year'},
 {'source_year': 2023,
  'rank': 4,
  'list_title': 'Happy Place',
  'book_url': 'https://www.goodreads.com/book/show/61718053-happy-place',
  'source': 'Goodreads Popular by Year'},
 {'source_year': 2023,
  'rank': 5,
  'list_title': 'Powerless (The Powerless Trilogy, #1)',
  'book_url': 'https://www.goodreads.com/book/show/75513900-powerless',
  'source': 'Goodreads Po

* Given that everything is working. Now we continue with the rest of the scrapping!

#### Scrape top 10 book links from 1975 to 2024

In [26]:
start_year = 1975
end_year = 2024
books_per_year = 10

book_links_data = []
failed_years = []

for year in range(start_year, end_year + 1):
    print(f"Extracting books from year {year}...")
    
    books = extract_books_from_year_page(year, books_per_year)
    
    if len(books) == 0:
        print(f"No books found for year {year}")
        failed_years.append(year)
    else:
        book_links_data.extend(books)
    
    time.sleep(3)

Extracting books from year 1975...
Extracting books from year 1976...
Extracting books from year 1977...
Extracting books from year 1978...
Extracting books from year 1979...
Extracting books from year 1980...
Extracting books from year 1981...
Extracting books from year 1982...
Extracting books from year 1983...
Extracting books from year 1984...
Extracting books from year 1985...
Extracting books from year 1986...
Extracting books from year 1987...
Extracting books from year 1988...
Extracting books from year 1989...
Extracting books from year 1990...
Extracting books from year 1991...
Extracting books from year 1992...
Extracting books from year 1993...
Extracting books from year 1994...
Extracting books from year 1995...
Extracting books from year 1996...
Extracting books from year 1997...
Extracting books from year 1998...
Extracting books from year 1999...
Extracting books from year 2000...
Extracting books from year 2001...
Extracting books from year 2002...
Extracting books fro

In [27]:
df_book_links = pd.DataFrame(book_links_data)
df_book_links.head()

,source_year,rank,list_title,book_url,source
0,1975,1,’Salem’s Lot,https://www.goodreads.com/book/show/11590._Sal...,Goodreads Popular by Year
1,1975,2,Tuck Everlasting,https://www.goodreads.com/book/show/84981.Tuck...,Goodreads Popular by Year
2,1975,3,"Shōgun (Asian Saga, #1)",https://www.goodreads.com/book/show/52382796-s...,Goodreads Popular by Year
3,1975,4,Factotum,https://www.goodreads.com/book/show/497199.Fac...,Goodreads Popular by Year
4,1975,5,Discipline and Punish: The Birth of the Prison,https://www.goodreads.com/book/show/80369.Disc...,Goodreads Popular by Year


In [28]:
df_book_links["list_title"].value_counts()

list_title
ACE #1                                           4
The BFG                                          2
Matilda                                          2
’Salem’s Lot                                     1
Tuck Everlasting                                 1
                                                ..
Just for the Summer (Part of Your World, #3)     1
The Housemaid is Watching (The Housemaid, #3)    1
Quicksilver (Fae & Alchemy, #1)                  1
The Teacher                                      1
The Boyfriend                                    1
Name: count, Length: 495, dtype: int64

NOTE: There are errors on some title of the books

In [29]:
df_book_links.to_csv("data/raw/goodreads_book_links_by_year.csv", index=False)

* Helper functions for cleaning detail-page values

In [30]:
def clean_count(text):
    """
    Converts strings like '3,461,396 ratings' into 3461396.
    """
    if text is None:
        return None
    
    match = re.search(r"([\d,]+)", text)
    
    if match:
        return int(match.group(1).replace(",", ""))
    
    return None


def clean_average_rating(text):
    """
    Extracts rating like 4.16 from text.
    """
    if text is None:
        return None
    
    match = re.search(r"(\d+\.\d+)", text)
    
    if match:
        return float(match.group(1))
    
    return None


def clean_pages(text):
    """
    Extracts number of pages from text like '336 pages, Hardcover'.
    """
    if text is None:
        return None
    
    match = re.search(r"(\d+)\s+pages", text)
    
    if match:
        return int(match.group(1))
    
    return None

#### Extracting detailed metadata from one book page

In [31]:
def extract_book_details(book_url):
    """
    Extracts detailed metadata from a Goodreads book page.
    """
    soup = get_soup(book_url)
    
    if soup is None:
        return None
    
    # Title from detail page
    title_tag = soup.select_one('h1[data-testid="bookTitle"]')
    
    if title_tag:
        title = title_tag.get_text(" ", strip=True)
    else:
        title_tag = soup.find("h1")
        title = title_tag.get_text(" ", strip=True) if title_tag else None
    
    # Author
    author_tag = soup.select_one('a[href*="/author/show/"]')
    author = author_tag.get_text(" ", strip=True) if author_tag else None
    
    # Average rating
    average_rating = None
    rating_tag = soup.select_one("div.RatingStatistics__rating")
    
    if rating_tag:
        try:
            average_rating = float(rating_tag.get_text(strip=True))
        except ValueError:
            average_rating = None
    
    # Ratings and reviews count
    ratings_count = None
    reviews_count = None
    
    stats_container = soup.select_one("div.BookPageMetadataSection__ratingStats")
    
    if stats_container:
        stats_text = stats_container.get_text(" ", strip=True)
    else:
        stats_text = soup.get_text(" ", strip=True)
    
    ratings_match = re.search(r"([\d,]+)\s+ratings", stats_text)
    reviews_match = re.search(r"([\d,]+)\s+reviews", stats_text)
    
    if ratings_match:
        ratings_count = int(ratings_match.group(1).replace(",", ""))
    
    if reviews_match:
        reviews_count = int(reviews_match.group(1).replace(",", ""))
    
    # Description
    description = None
    description_tag = soup.select_one('[data-testid="description"]')
    
    if description_tag:
        description = description_tag.get_text(" ", strip=True)
    
    # Genres
    genre_tags = soup.select('a[href*="/genres/"]')
    genres = list(dict.fromkeys([
        g.get_text(" ", strip=True) 
        for g in genre_tags 
        if g.get_text(strip=True)
    ]))
    
    # Pages and publication date
    pages = None
    published_date = None
    
    details_text = soup.get_text(" ", strip=True)
    
    pages_match = re.search(r"(\d+)\s+pages", details_text)
    published_match = re.search(
        r"First published\s+([A-Za-z]+\s+\d{1,2},\s+\d{4}|[A-Za-z]+\s+\d{4}|\d{4})", 
        details_text
    )
    
    if pages_match:
        pages = int(pages_match.group(1))
    
    if published_match:
        published_date = published_match.group(1)
    
    # Image URL
    image_url = None
    image_tag = soup.select_one('img.ResponsiveImage')
    
    if image_tag:
        image_url = image_tag.get("src")
    
    return {
        "title": title,
        "author": author,
        "average_rating": average_rating,
        "ratings_count": ratings_count,
        "reviews_count": reviews_count,
        "description": description,
        "genres": ", ".join(genres) if genres else None,
        "pages": pages,
        "published_date": published_date,
        "image_url": image_url
    }

In [32]:
test_url = df_book_links.loc[0, "book_url"]

test_details = extract_book_details(test_url)

test_details

{'title': '’Salem’s Lot',
 'author': 'Stephen  King',
 'average_rating': 4.1,
 'ratings_count': 674093,
 'reviews_count': 28812,
 'description': "Librarian's Note: Alternate-cover edition for ISBN 0450031063 Thousands of miles away from the small township of 'Salem's Lot, two terrified people, a man and a boy, still share the secrets of those clapboard houses and tree-lined streets. They must return to 'Salem's Lot for a final confrontation with the unspeakable evil that lives on in the town.",
 'genres': 'Horror, Fiction, Vampires, Fantasy, Thriller, Paranormal, Audiobook',
 'pages': 483,
 'published_date': 'October 17, 1975',
 'image_url': 'https://m.media-amazon.com/images/S/compressed.photo.goodreads.com/books/1738025067i/11590.jpg'}

#### Final scrapping including all details

In [33]:
detailed_books = []
failed_books = []

for index, row in df_book_links.iterrows():
    print(f"Scraping details {index + 1}/{len(df_book_links)}: {row['book_url']}")
    
    details = extract_book_details(row["book_url"])
    
    if details is None:
        failed_books.append(row["book_url"])
        continue
    
    combined_data = {
        "source_year": row["source_year"],
        "rank": row["rank"],
        "list_title": row["list_title"],
        "book_url": row["book_url"],
        "source": row["source"]
    }
    
    combined_data.update(details)
    detailed_books.append(combined_data)
    
    time.sleep(4)

Scraping details 1/500: https://www.goodreads.com/book/show/11590._Salem_s_Lot
Scraping details 2/500: https://www.goodreads.com/book/show/84981.Tuck_Everlasting
Scraping details 3/500: https://www.goodreads.com/book/show/52382796-sh-gun
Scraping details 4/500: https://www.goodreads.com/book/show/497199.Factotum
Scraping details 5/500: https://www.goodreads.com/book/show/80369.Discipline_and_Punish
Scraping details 6/500: https://www.goodreads.com/book/show/40881649-crocodile-on-the-sandbank
Scraping details 7/500: https://www.goodreads.com/book/show/37743.Forever_
Scraping details 8/500: https://www.goodreads.com/book/show/43339.Where_Are_the_Children_
Scraping details 9/500: https://www.goodreads.com/book/show/95747.The_Miracle_of_Mindfulness
Scraping details 10/500: https://www.goodreads.com/book/show/6690.Danny_the_Champion_of_the_World
Scraping details 11/500: https://www.goodreads.com/book/show/43763.Interview_with_the_Vampire
Scraping details 12/500: https://www.goodreads.com/bo

In [34]:
df2 = pd.DataFrame(detailed_books)

df2.head()

,source_year,rank,list_title,book_url,source,title,author,average_rating,ratings_count,reviews_count,description,genres,pages,published_date,image_url
0,1975,1,’Salem’s Lot,https://www.goodreads.com/book/show/11590._Sal...,Goodreads Popular by Year,’Salem’s Lot,Stephen King,4.10,674093,28812,Librarian's Note: Alternate-cover edition for ...,"Horror, Fiction, Vampires, Fantasy, Thriller, ...",483,"October 17, 1975",https://m.media-amazon.com/images/S/compressed...
1,1975,2,Tuck Everlasting,https://www.goodreads.com/book/show/84981.Tuck...,Goodreads Popular by Year,Tuck Everlasting,Natalie Babbitt,3.91,299733,13997,Introduced by New York Times –bestselling auth...,"Fantasy, Classics, Young Adult, Fiction, Child...",148,"January 1, 1975",https://m.media-amazon.com/images/S/compressed...
2,1975,3,"Shōgun (Asian Saga, #1)",https://www.goodreads.com/book/show/52382796-s...,Goodreads Popular by Year,Shōgun,James Clavell,4.41,217483,9063,After Englishman John Blackthorne is lost at s...,"Historical Fiction, Fiction, Japan, Historical...",1152,"January 1, 1975",https://m.media-amazon.com/images/S/compressed...
3,1975,4,Factotum,https://www.goodreads.com/book/show/497199.Fac...,Goodreads Popular by Year,Factotum,Charles Bukowski,3.92,77117,3254,"One of Bukowski's best, this beer-soaked, deli...","Fiction, Classics, Novels, Literature, America...",205,"January 1, 1975",https://m.media-amazon.com/images/S/compressed...
4,1975,5,Discipline and Punish: The Birth of the Prison,https://www.goodreads.com/book/show/80369.Disc...,Goodreads Popular by Year,Discipline and Punish: The Birth of the Prison,Michel Foucault,4.23,37806,1914,Librarian note: an alternate cover for this ed...,"Philosophy, Nonfiction, History, Sociology, Th...",333,"January 1, 1975",https://m.media-amazon.com/images/S/compressed...


In [35]:
df2.to_csv("data/raw/goodreads_books_detailed1.csv", index=False)

* Cleaning data and columns

In [9]:
df2 = pd.read_csv("data/raw/goodreads_books_detailed1.csv")
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   source_year     500 non-null    int64  
 1   rank            500 non-null    int64  
 2   list_title      500 non-null    str    
 3   book_url        500 non-null    str    
 4   source          500 non-null    str    
 5   title           500 non-null    str    
 6   author          500 non-null    str    
 7   average_rating  500 non-null    float64
 8   ratings_count   500 non-null    int64  
 9   reviews_count   500 non-null    int64  
 10  description     500 non-null    str    
 11  genres          500 non-null    str    
 12  pages           500 non-null    int64  
 13  published_date  500 non-null    str    
 14  image_url       500 non-null    str    
dtypes: float64(1), int64(5), str(9)
memory usage: 681.6 KB


In [ ]:
df2 = df2.drop(columns=["list_title", "rank", "published_date"]) # dropping columns not needed
df2["title"].value_counts().head(20)  #  There are some duplicate books. We have to drop them!

title
The Host                                                3
The BFG                                                 2
Pet Sematary                                            2
The Witches                                             2
If You Give a Mouse a Cookie                            2
Matilda                                                 2
The Pillars of the Earth                                2
The Firm                                                2
The Last Wish                                           2
The Power of Now: A Guide to Spiritual Enlightenment    2
City of Ashes                                           2
City of Glass                                           2
The Song of Achilles                                    2
Shadow and Bone                                         2
The Wedding People                                      2
’Salem’s Lot                                            1
Tuck Everlasting                                        1
Shōgun  

In [ ]:
duplicate_titles = df2[df2["title"].duplicated(keep=False)].sort_values("title")
duplicate_titles[["title", "author", "source_year", "average_rating", "ratings_count", "source"]].head(29)  # For sure we have to drop them

,title,author,source_year,average_rating,ratings_count,source
335,City of Ashes,Cassandra Clare,2008,4.10,1004685,Goodreads Popular by Year
336,City of Ashes,Cassandra Clare,2008,4.10,1004685,Goodreads Popular by Year
347,City of Glass,Cassandra Clare,2009,4.27,1023498,Goodreads Popular by Year
348,City of Glass,Cassandra Clare,2009,4.27,1023498,Goodreads Popular by Year
108,If You Give a Mouse a Cookie,Laura Joffe Numeroff,1985,4.30,317240,Goodreads Popular by Year
109,If You Give a Mouse a Cookie,Laura Joffe Numeroff,1985,4.30,317240,Goodreads Popular by Year
131,Matilda,Roald Dahl,1988,4.34,1133306,Goodreads Popular by Year
133,Matilda,Roald Dahl,1988,4.34,1133306,Goodreads Popular by Year
80,Pet Sematary,Stephen King,1983,4.09,718110,Goodreads Popular by Year
81,Pet Sematary,Stephen King,1983,4.09,718110,Goodreads Popular by Year


In [15]:
df2_clean = df2.drop_duplicates(
    subset=["title", "author"],
    keep="first"
).reset_index(drop=True)

In [16]:
df2_clean[["title", "author"]].duplicated().sum()

np.int64(0)

In [17]:
df2_clean["title"].value_counts().head(20) 

title
The Firm                                                                     2
’Salem’s Lot                                                                 1
Tuck Everlasting                                                             1
Shōgun                                                                       1
Factotum                                                                     1
Discipline and Punish: The Birth of the Prison                               1
Crocodile on the Sandbank                                                    1
Forever...                                                                   1
Where Are the Children?                                                      1
The Miracle of Mindfulness: An Introduction to the Practice of Meditation    1
Danny the Champion of the World                                              1
Interview with the Vampire                                                   1
The Selfish Gene                              

In [ ]:
df2_clean = df2_clean[df2_clean["description"] != "1"].copy() #  Droppping a description == 1

* Saving Clean dataset

In [19]:
df2_clean.to_csv("data/GoodBooks_clean.csv", index=False)